# Predicting Housing Prices in Buenos Aires

This notebook covers data preparation, cleaning, and exploratory data analysis. Modeling begins after this stage.

## Import Libraries

Import the libraries needed for data preparation and EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# 1. Prepare Data

## 1.1 Import Data

Load both Buenos Aires real-estate extracts and combine them into one dataframe.

In [ ]:
file_1 = "data/buenos-aires-real-estate-1.csv"
file_2 = "data/buenos-aires-real-estate-2.csv"

df1 = pd.read_csv(file_1)
df2 = pd.read_csv(file_2)
df = pd.concat([df1, df2], ignore_index=True)

print(f"File 1: {df1.shape}")
print(f"File 2: {df2.shape}")
print(f"Combined dataframe: {df.shape}")

In [ ]:
df.head()

## 1.2 Initial Inspection

Check column types, duplicate rows, operations, property types, and target availability before cleaning.

In [ ]:
df.info()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("\nOperations:\n", df["operation"].value_counts(dropna=False))
print("\nProperty types:\n", df["property_type"].value_counts(dropna=False))
print("\nMissing target values:", df["price_aprox_usd"].isna().sum())

# 2. Data Cleaning

The cleaning choices below remove invalid records without discarding useful rows solely because optional fields are missing.

## 2.1 Remove Exact Duplicates

Remove exact duplicate rows after combining the files.

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
print(f"Removed {before - len(df):,} duplicate rows")

## 2.2 Define the Analysis Scope

Keep sale listings with a positive USD price. Records without a target cannot be used for price analysis.

In [ ]:
df["price_aprox_usd"] = pd.to_numeric(df["price_aprox_usd"], errors="coerce")
df = df.loc[
    (df["operation"].eq("sell"))
    & (df["price_aprox_usd"].notna())
    & (df["price_aprox_usd"] > 0)
].copy()

print(f"Rows after scope filter: {len(df):,}")
print(f"Price range: ${df['price_aprox_usd'].min():,.0f} to ${df['price_aprox_usd'].max():,.0f}")

## 2.3 Convert Numeric Fields

Convert quantitative columns explicitly so summaries and plots use numeric values.

In [ ]:
numeric_columns = [
    "price", "price_aprox_local_currency", "price_aprox_usd",
    "surface_total_in_m2", "surface_covered_in_m2",
    "price_usd_per_m2", "price_per_m2", "floor", "rooms", "expenses"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

## 2.4 Handle Impossible Measurements

Treat zero or negative areas, room counts, and expenses as missing rather than valid measurements. Missing values are retained for later imputation decisions.

In [ ]:
nonnegative_columns = [
    "surface_total_in_m2", "surface_covered_in_m2",
    "rooms", "expenses"
]

for column in nonnegative_columns:
    df.loc[df[column] <= 0, column] = np.nan

df.loc[df["floor"] <= 0, "floor"] = np.nan

# Covered area should not exceed total area when both values are available.
invalid_area = (
    df["surface_total_in_m2"].notna()
    & df["surface_covered_in_m2"].notna()
    & (df["surface_covered_in_m2"] > df["surface_total_in_m2"])
)
df.loc[invalid_area, "surface_covered_in_m2"] = np.nan

print(f"Invalid area relationships corrected: {invalid_area.sum():,}")

## 2.5 Extract Geographic Features

Split coordinates and the hierarchical location field into analysis-friendly columns.

In [ ]:
coordinates = df["lat-lon"].str.split(",", n=1, expand=True)
df["latitude"] = pd.to_numeric(coordinates[0], errors="coerce")
df["longitude"] = pd.to_numeric(coordinates[1], errors="coerce")

location_parts = df["place_with_parent_names"].fillna("").str.strip("|").str.split("|")
df["province"] = location_parts.str[1]
df["municipality"] = location_parts.str[2]
df["neighborhood"] = location_parts.str[3]

df[["place_with_parent_names", "province", "municipality", "neighborhood", "latitude", "longitude"]].head()

## 2.6 Review Missingness

Do not drop all incomplete rows: floor, expenses, rooms, and coordinates are optional listing attributes.

In [ ]:
missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .rename("missing_percent")
    .to_frame()
)
missing.head(12)

## 2.7 Modeling-Ready Feature Note

`price_usd_per_m2` and `price_per_m2` are derived from price and area. They are useful for EDA but must be excluded from future price models to prevent target leakage.

In [ ]:
leakage_columns = ["price_usd_per_m2", "price_per_m2"]
modeling_exclusions = leakage_columns + ["price_aprox_usd", "properati_url"]
print("Exclude from future predictors:", modeling_exclusions)

# 3. Exploratory Data Analysis

Explore price distributions, property mix, size relationships, geographic variation, and missing data.

## 3.1 Descriptive Statistics

Review the cleaned target and main numeric variables.

In [ ]:
eda_columns = [
    "price_aprox_usd", "surface_total_in_m2", "surface_covered_in_m2",
    "rooms", "floor", "expenses", "latitude", "longitude"
]
df[eda_columns].describe().T

## 3.2 Price Distribution

Housing prices are right-skewed, so inspect both the original and log-transformed scales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["price_aprox_usd"], bins=50, ax=axes[0])
axes[0].set_title("Listing Price Distribution")
axes[0].set_xlabel("Price (USD)")

sns.histplot(np.log1p(df["price_aprox_usd"]), bins=50, ax=axes[1])
axes[1].set_title("Log Price Distribution")
axes[1].set_xlabel("log(Price + 1)")

plt.tight_layout()

## 3.3 Property-Type Mix and Prices

Compare the number of listings and price distributions across property types.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = df["property_type"].value_counts().index
sns.countplot(data=df, y="property_type", order=order, ax=axes[0])
axes[0].set_title("Listings by Property Type")

plot_df = df.assign(price_plot=df["price_aprox_usd"].clip(upper=df["price_aprox_usd"].quantile(0.99)))
sns.boxplot(data=plot_df, x="property_type", y="price_plot", ax=axes[1])
axes[1].set_title("Price by Property Type")
axes[1].set_xlabel("")
axes[1].set_ylabel("Price (USD, capped at 99th percentile)")

plt.tight_layout()

## 3.4 Price and Property Size

Examine whether larger properties generally have higher listing prices.

In [ ]:
plot_df = df[["surface_total_in_m2", "price_aprox_usd", "property_type"]].dropna()
plot_df = plot_df[plot_df["surface_total_in_m2"] > 0]
plot_df = plot_df[plot_df["price_aprox_usd"] <= plot_df["price_aprox_usd"].quantile(0.99)]

sns.scatterplot(
    data=plot_df.sample(min(3000, len(plot_df)), random_state=42),
    x="surface_total_in_m2",
    y="price_aprox_usd",
    hue="property_type",
    alpha=0.6
)
plt.title("Price versus Total Surface Area")
plt.xlabel("Total surface area (m²)")
plt.ylabel("Price (USD)")
plt.legend(title="Property type")
plt.show()

## 3.5 Geographic Price Patterns

Compare median prices across neighborhoods with enough observations for a stable summary.

In [ ]:
neighborhood_summary = (
    df.dropna(subset=["neighborhood"])
    .groupby("neighborhood")
    .agg(listings=("price_aprox_usd", "size"), median_price=("price_aprox_usd", "median"))
    .query("listings >= 20")
    .sort_values("median_price", ascending=False)
    .head(15)
    .sort_values("median_price")
)

neighborhood_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=neighborhood_summary,
    x="median_price",
    y=neighborhood_summary.index,
    color="steelblue"
)
plt.title("Top Neighborhoods by Median Listing Price")
plt.xlabel("Median price (USD)")
plt.ylabel("Neighborhood")
plt.tight_layout()

## 3.6 Missing-Data Profile

Visualize missingness to guide the later preprocessing and imputation strategy.

In [ ]:
missing_plot = missing.head(12).sort_values("missing_percent")

plt.figure(figsize=(10, 6))
sns.barplot(
    data=missing_plot,
    x="missing_percent",
    y=missing_plot.index,
    color="darkorange"
)
plt.title("Missing Values by Column")
plt.xlabel("Missing values (%)")
plt.ylabel("")
plt.tight_layout()

## EDA Summary

The cleaned dataframe is now ready for the next stage: feature selection, train/test splitting, preprocessing pipelines, and regression modeling.